# reconcile_silver
Bronze/Silver/Reject 건수 항등식 검증(설계서 Part 15). 이 식이 안 맞으면 데이터가 어느 단계에서 조용히 사라진 것이므로, 통과하지 못하면 **실패로 종료**해 `JOB_RETRY_REPLAY` 대상이 되게 한다.

```
GHSA: Bronze = Silver(valid) + Reject
EPSS: Bronze = Silver(valid) + Reject   (+ active_cve_set 밖이라 추가로 제외된 건수는 별도 정보성 지표로 기록)
```

**주의**: 이 노트북은 `validate_ghsa`/`validate_epss`와 **같은 `batch_id`를 파이프라인으로부터 전달받아야** Reject 건수를 올바른 배치로 집계할 수 있다(§Part 7 `meta_batch_run` 설계와 동일 원칙). KEV는 이벤트 단위로 처리되고 `batch_id`가 파일명 기준 결정적 값으로 바뀌었기 때문에 이 항등식 검증 대상에 포함하지 않는다 — Reject는 `reject_kev`에 그대로 쌓이고, 재처리는 `JOB_RETRY_REPLAY`가 개별 이벤트 단위로 처리한다.

In [ ]:
%run 00_silver_common

In [ ]:
def count_json_flex(path: str) -> int:
    try:
        df = spark.read.option("multiLine", "true").json(path)
        if len(df.columns) > 0:
            return df.count()
    except Exception:
        pass
    try:
        return spark.read.json(path).count()
    except Exception:
        return 0

def count_reject(table_key: str) -> int:
    if not table_exists(TBL[table_key]):
        return 0
    return spark.table(TBL[table_key]).filter(F.col("batch_id") == batch_id).count()


### GHSA Reconciliation

In [ ]:
bronze_ghsa_count = count_json_flex(f"{BRONZE_GHSA_PATH}/{snapshot_date}/*")
reject_ghsa_count = count_reject("reject_ghsa")
valid_ghsa_count = spark.table(TBL["stg_ghsa_valid"]).count() if table_exists(TBL["stg_ghsa_valid"]) else 0

ghsa_reconciled = bronze_ghsa_count == (valid_ghsa_count + reject_ghsa_count)

log_dq_metric(
    "reconcile_silver", "ghsa_reconciliation", bronze_ghsa_count, passed=ghsa_reconciled,
    detail=f"bronze={bronze_ghsa_count}, valid={valid_ghsa_count}, reject={reject_ghsa_count}",
)
print(f"[reconcile_silver][GHSA] bronze={bronze_ghsa_count} valid={valid_ghsa_count} "
      f"reject={reject_ghsa_count} reconciled={ghsa_reconciled}")


### EPSS Reconciliation

In [ ]:
epss_dir = f"{BRONZE_EPSS_PATH}/{snapshot_date}"
bronze_epss_count = (
    spark.read.option("header", "true").option("comment", "#").csv(f"{epss_dir}/*.csv.gz").count()
)
reject_epss_count = count_reject("reject_epss")
valid_epss_count = spark.table(TBL["stg_epss_valid"]).count() if table_exists(TBL["stg_epss_valid"]) else 0

epss_reconciled = bronze_epss_count == (valid_epss_count + reject_epss_count)

# active_cve_set 밖이라 silver_epss에는 최종적으로 안 들어간 건수 (정보성 지표, 실패 조건 아님)
silver_epss_today_count = 0
if table_exists(TBL["epss"]):
    silver_epss_today_count = spark.table(TBL["epss"]).filter(F.col("batch_id") == batch_id).count()
excluded_by_active_set = max(valid_epss_count - silver_epss_today_count, 0)

log_dq_metric(
    "reconcile_silver", "epss_reconciliation", bronze_epss_count, passed=epss_reconciled,
    detail=f"bronze={bronze_epss_count}, valid={valid_epss_count}, reject={reject_epss_count}, "
           f"excluded_by_active_set={excluded_by_active_set}",
)
print(f"[reconcile_silver][EPSS] bronze={bronze_epss_count} valid={valid_epss_count} "
      f"reject={reject_epss_count} excluded_by_active_set={excluded_by_active_set} "
      f"reconciled={epss_reconciled}")


In [ ]:
all_reconciled = ghsa_reconciled and epss_reconciled

if all_reconciled:
    log_batch_run("reconcile_silver", "SUCCESS")
    print(f"[reconcile_silver] PASS batch_id={batch_id}")
else:
    log_batch_run(
        "reconcile_silver", "FAILED",
        error_message=f"ghsa_reconciled={ghsa_reconciled}, epss_reconciled={epss_reconciled}",
    )
    raise RuntimeError(
        f"Reconciliation FAILED: ghsa_reconciled={ghsa_reconciled}, epss_reconciled={epss_reconciled}. "
        f"batch_id={batch_id} -- JOB_RETRY_REPLAY 대상."
    )
